# Project 02: GitHub + Drive + Colab GPU

권장 구조입니다.

```text
코드   -> GitHub
데이터 -> Google Drive
실행   -> Colab GPU
편집   -> VS Code
```

로컬 VS Code에서 코드를 수정하고 GitHub에 push한 뒤, Colab에서는 `git pull`로 최신 코드를 받아 GPU 학습만 진행합니다. 학습 결과는 Drive에 저장합니다.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. Google Drive 연결

데이터 zip, checkpoint, sample, FID 결과는 Drive에 보관합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. GitHub에서 코드 가져오기

`REPO_URL`을 본인 GitHub 저장소 주소로 바꾸세요. 공개 저장소면 그대로 clone됩니다. 비공개 저장소라면 Colab의 Secret 또는 GitHub token 설정이 필요합니다.

In [ ]:
from pathlib import Path

# TODO: 본인 GitHub 저장소 주소로 수정
REPO_URL = 'https://github.com/your-id/project2.git'
BRANCH = 'main'

PROJECT_DIR = Path('/content/project2')
if PROJECT_DIR.exists():
    %cd {PROJECT_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

!git log -1 --oneline
!ls

## 3. Drive 경로 설정

Drive에는 큰 파일만 둡니다.

권장 구조:

```text
MyDrive/project2_data/
  train_50k_512.zip
  train_50k_1024.zip
  train_50k_256.zip       # 선택: 256부터 추가 학습할 때 사용
  valid_10k_256.zip       # 선택
  valid_10k_512.zip       # 선택
  valid_10k_1024.zip      # FID용 validation real image zip

MyDrive/project2_outputs/
  pg_512/
  pg_1024/
  quality_1024/
  eval_runs/
```

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_ROOT = DRIVE_ROOT / 'project2_data'
OUTPUT_ROOT = DRIVE_ROOT / 'project2_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_ZIP_512 = DATA_ROOT / 'train_50k_512.zip'
TRAIN_ZIP_1024 = DATA_ROOT / 'train_50k_1024.zip'
REAL_ZIP_1024 = DATA_ROOT / 'valid_10k_1024.zip'

print('DATA_ROOT =', DATA_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('TRAIN_ZIP_512 exists:', TRAIN_ZIP_512.exists(), TRAIN_ZIP_512)
print('TRAIN_ZIP_1024 exists:', TRAIN_ZIP_1024.exists(), TRAIN_ZIP_1024)
print('REAL_ZIP_1024 exists:', REAL_ZIP_1024.exists(), REAL_ZIP_1024)

## 4. 의존성 설치

In [ ]:
!pip -q install -r requirements.txt
!pip -q install pytorch-fid scipy

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 5. Config를 Drive 경로로 패치

GitHub 코드는 작게 유지하고, 실제 데이터 경로와 실험 출력 경로만 Colab에서 주입합니다.

In [ ]:
import yaml

def patch_config(config_path, train_zip, run_name):
    config_path = Path(config_path)
    with config_path.open('r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    cfg['training']['train_zip'] = str(train_zip)
    cfg.setdefault('out', {})['run_dir'] = str(OUTPUT_ROOT / run_name)
    with config_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    print(config_path, '->', cfg['training']['train_zip'], cfg['out']['run_dir'])

patch_config('configs/baseline_512.yaml', TRAIN_ZIP_512, 'pg_512')
patch_config('configs/baseline_1024.yaml', TRAIN_ZIP_1024, 'pg_1024')
patch_config('configs/quality_1024.yaml', TRAIN_ZIP_1024, 'quality_1024')

## 6. 모델 크기 확인

Generator는 과제 조건상 40M 미만이어야 합니다.

In [ ]:
!python count_params.py --config configs/baseline_1024.yaml
!python count_params.py --config configs/quality_1024.yaml

## 7. Baseline checkpoint 확인

In [ ]:
from IPython.display import Image, display

!python generate.py --ckpt ckpt/ffhq256_baseline.pt --out sample_256.png --n 16 --nrow 4 --batch-size 8
display(Image('sample_256.png'))

## 8. 512 Stage 학습

처음에는 smoke test로 저장/샘플링이 되는지 확인합니다. 실제 학습 셀은 주석을 풀고 실행하세요.

In [ ]:
!python train.py --config configs/baseline_512.yaml --init-from ckpt/ffhq256_baseline.pt --total-images 1024

In [ ]:
# 실제 512 학습
# !python train.py --config configs/baseline_512.yaml --init-from ckpt/ffhq256_baseline.pt

## 9. 재개용 checkpoint 탐색

In [ ]:
def latest_ckpt(run_dir):
    run_dir = Path(run_dir)
    candidates = sorted(run_dir.glob('ckpt_*.pt'))
    if (run_dir / 'final.pt').exists():
        candidates.append(run_dir / 'final.pt')
    return candidates[-1] if candidates else None

latest_512 = latest_ckpt(OUTPUT_ROOT / 'pg_512')
print('latest_512 =', latest_512)

In [ ]:
# 512 재개
# !python train.py --config configs/baseline_512.yaml --resume "{latest_512}"

## 10. 1024 Stage 학습

512 checkpoint에서 1024 모델로 warm-start합니다.

In [ ]:
INIT_1024 = OUTPUT_ROOT / 'pg_512' / 'final.pt'
if not INIT_1024.exists():
    INIT_1024 = latest_ckpt(OUTPUT_ROOT / 'pg_512')
print('INIT_1024 =', INIT_1024)
assert INIT_1024 is not None and INIT_1024.exists(), '먼저 512 checkpoint를 만들어야 합니다.'

In [ ]:
# baseline 1024 smoke test
!python train.py --config configs/baseline_1024.yaml --init-from "{INIT_1024}" --total-images 256

In [ ]:
# 실제 baseline 1024 학습
# !python train.py --config configs/baseline_1024.yaml --init-from "{INIT_1024}"

## 11. Quality 1024 비교 실험

기본 1024보다 채널을 조금 키운 실험입니다. 같은 512 checkpoint에서 시작해서 FID와 샘플을 비교하세요.

In [ ]:
# quality 1024 smoke test
!python train.py --config configs/quality_1024.yaml --init-from "{INIT_1024}" --total-images 256

In [ ]:
# 실제 quality 1024 학습
# !python train.py --config configs/quality_1024.yaml --init-from "{INIT_1024}"

## 12. Checkpoint 비교 평가

빠른 비교는 `--n 128`, 최종 비교는 `--n 1000~5000` 정도로 늘리세요. validation real image 폴더가 있으면 FID가 계산됩니다.

In [ ]:
ckpts = sorted(OUTPUT_ROOT.glob('*/ckpt_*.pt'))[-8:]
if (OUTPUT_ROOT / 'pg_1024' / 'final.pt').exists():
    ckpts.append(OUTPUT_ROOT / 'pg_1024' / 'final.pt')
if (OUTPUT_ROOT / 'quality_1024' / 'final.pt').exists():
    ckpts.append(OUTPUT_ROOT / 'quality_1024' / 'final.pt')
ckpt_args = ' '.join(f'"{p}"' for p in ckpts)
real_arg = f'--real-zip "{REAL_ZIP_1024}"' if REAL_ZIP_1024.exists() else ''
print('ckpts:', ckpts)
print('real_arg:', real_arg)

In [ ]:
!python eval_checkpoints.py --ckpts {ckpt_args} {real_arg} --out-dir "{OUTPUT_ROOT / 'eval_quick'}" --n 128 --batch-size 4

In [ ]:
import pandas as pd
results = pd.read_csv(OUTPUT_ROOT / 'eval_quick' / 'fid_results.csv')
display(results)
BEST_CKPT = Path(results.iloc[0]['checkpoint']) if len(results) else latest_ckpt(OUTPUT_ROOT / 'pg_1024')
print('BEST_CKPT =', BEST_CKPT)

In [ ]:
# 최종 후보만 오래 평가
# !python eval_checkpoints.py --ckpts "{OUTPUT_ROOT / 'pg_1024' / 'final.pt'}" "{OUTPUT_ROOT / 'quality_1024' / 'final.pt'}" --real-zip "{REAL_ZIP_1024}" --out-dir "{OUTPUT_ROOT / 'eval_final'}" --n 5000 --batch-size 4

## 13. 최종 샘플 확인

In [ ]:
assert BEST_CKPT is not None and Path(BEST_CKPT).exists(), 'BEST_CKPT가 필요합니다.'
!python generate.py --ckpt "{BEST_CKPT}" --out sample_best.png --n 16 --nrow 4 --batch-size 4
display(Image('sample_best.png'))

## 14. ONNX Export 및 검증

In [ ]:
!python export_onnx.py --ckpt "{BEST_CKPT}" --out "{OUTPUT_ROOT / 'submission.onnx'}" --batch-size 1

In [ ]:
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession(str(OUTPUT_ROOT / 'submission.onnx'), providers=['CPUExecutionProvider'])
out = sess.run(None, {'z': np.random.randn(1, 512).astype(np.float32)})[0]
print(out.shape, out.dtype, out.min(), out.max())
assert out.shape == (1, 3, 1024, 1024)

## 15. 제출 ZIP 생성

보고서 PDF는 Drive나 현재 런타임에 올린 뒤 `REPORT_PDF` 경로만 맞추세요.

In [ ]:
STUDENT_ID = '2025xxxxxx'
STUDENT_NAME = '홍길동'
REPORT_PDF = DRIVE_ROOT / f'{STUDENT_ID}_project02_report.pdf'

SUBMIT_DIR = Path(f'/content/{STUDENT_ID}_project02')
!rm -rf "{SUBMIT_DIR}"
(SUBMIT_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)

!cp -r src "{SUBMIT_DIR}/src"
!cp README.md "{SUBMIT_DIR}/README.md"
!cp "{BEST_CKPT}" "{SUBMIT_DIR}/checkpoints/model.pth"
!cp "{OUTPUT_ROOT / 'submission.onnx'}" "{SUBMIT_DIR}/submission.onnx"
if REPORT_PDF.exists():
    !cp "{REPORT_PDF}" "{SUBMIT_DIR}/{STUDENT_ID}_project02_report.pdf"
else:
    print('보고서 PDF가 아직 없습니다:', REPORT_PDF)

ZIP_PATH = OUTPUT_ROOT / f'{STUDENT_ID}_project02.zip'
!cd /content && zip -qr "{ZIP_PATH}" "{SUBMIT_DIR.name}"
print('created:', ZIP_PATH)

In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))